# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to explore and analyze the FAIR² clinical oncology dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library, referencing all record sets, fields, and columns by their `@id` fields for full reproducibility and traceability.

### Dataset Source
The dataset is described by a Croissant schema accessible at:
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load the dataset's Croissant schema metadata and inspect its contents using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata_obj = dataset.metadata
metadata_dict = metadata_obj.to_json()

print(f"Dataset Name: {metadata_obj.name}\n")
print(f"Description: {metadata_obj.description}\n")
print(f"Published: {getattr(metadata_obj, 'datePublished', 'Unknown')}")
print(f"Version: {getattr(metadata_obj, 'version', 'Unknown')}")
print(f"License: {getattr(metadata_obj, 'license', 'Unknown')}")

## 2. Data Overview

Let's enumerate the available record sets, along with their `@id` fields, and the fields they contain. This enables referencing entities by their `@id` according to FAIR principles.

In [ ]:
# List record sets with @id and their fields
print("Available record sets and their fields (by @id):\n")
record_sets = metadata_obj.recordSet
if isinstance(record_sets, dict):
    # In Croissant, it can be a dict if only one record set exists
    record_sets = [record_sets]

for rs in record_sets:
    print(f"Record set: {rs['@id']}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    if not fields:
        print("  (No fields listed, will infer from records)")
        continue
    for field in fields:
        print(f"  field @id: {field['@id']} -- name: {field.get('name', '')}")
    print()
if not record_sets:
    print("No record sets were found in the Croissant schema metadata. You may need to check the Croissant schema JSON for referenced tables/record sets.")

## 3. Data Extraction

Extract data from the record set(s) into Pandas DataFrames for analysis. All references to record sets and fields will use the `@id` provided in the Croissant schema.

In [ ]:
# If no record sets are explicitly listed in the metadata, we can still try typical ids or extract the first available
if record_sets and isinstance(record_sets, list):
    record_set_ids = [recset['@id'] for recset in record_sets]
else:
    # For datasets with a single implicit record set (common for one table CSVs)
    record_set_ids = ['cr:RecordSet']  # Default per Croissant vocabulary
print(f"Extracting from record sets: {record_set_ids}\n")

# Extract records for each record set via @id
dataframes = {}
for recset_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=recset_id))
        df = pd.DataFrame(records)
        dataframes[recset_id] = df
        print(f"Loaded DataFrame for {recset_id} with {len(df)} records and columns: {df.columns.tolist()}")
    except Exception as e:
        print(f"Could not load records for record set {recset_id}: {e}")
        continue

# Display the first few rows of the first record set
main_record_set_id = record_set_ids[0]
if main_record_set_id in dataframes:
    display(dataframes[main_record_set_id].head())
else:
    print("No DataFrame was loaded.")

## 4. Exploratory Data Analysis (EDA)

We'll filter and analyze the clinical data. For this, choose numeric and grouping fields using their `@id` present in the columns. Example: analyze 'Age' and group by 'Sex'.

> **NOTE:** Make sure to use the actual `@id` from the listing above as your DataFrame columns. Adjust the fields as necessary based on the printed column names.

In [ ]:
# Example: Assume typical Croissant column field naming: 'cr:Age', 'cr:Sex', etc.
df = dataframes[main_record_set_id]

# Find numeric columns by printing dtypes
print("Data types in main record set:")
print(df.dtypes)

# Set field IDs (adjust as revealed by above)
numeric_field_id = None
group_field_id = None
for c in df.columns:
    if 'age' in c.lower():
        numeric_field_id = c
    if 'sex' in c.lower() or 'gender' in c.lower():
        group_field_id = c
if numeric_field_id is None:
    print("Could not auto-detect an age-related column. Please inspect columns and set manually.")
    numeric_field_id = df.select_dtypes(include='number').columns[0]
if group_field_id is None:
    print("Could not auto-detect a sex/gender-related column. Please set manually.")

threshold = 50  # e.g., filter for patients over 50 years old
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold} (n={len(filtered_df)}):\n")
print(filtered_df[[numeric_field_id, group_field_id]].head())

# Normalize the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / (filtered_df[numeric_field_id].std() + 1e-8)

print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# If the group field exists, perform a group analysis
if group_field_id and group_field_id in df.columns:
    grouped_stats = filtered_df.groupby(group_field_id)[numeric_field_id].agg(['count','mean','std','min','max'])
    print(f"\nGrouped statistics by {group_field_id}:")
    print(grouped_stats)
else:
    print(f"No group field set or found for grouping.")

## 5. Visualization

Now, visualize the distribution of age and its relation to sex/grouping. This will help us understand demographic structure and any group differences. If other numeric fields are available, you might repeat this process with microsatellite instability (MSI) status, metastasis, etc.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,5))
sns.histplot(df[numeric_field_id].dropna(), bins=10, kde=True)
plt.title(f"Distribution of {numeric_field_id} (typically Age)")
plt.xlabel(numeric_field_id)
plt.ylabel('Frequency')
plt.show()

# Boxplot by group if available
if group_field_id and group_field_id in df.columns:
    plt.figure(figsize=(8,5))
    sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion

- We successfully loaded a deeply described clinical tabular dataset using `mlcroissant`, referencing all entities by their `@id` fields for reproducibility.
- We visualized and analyzed patient demographics, filtering and grouping by standard attributes such as age and sex. This approach can be extended to other fields (comorbidities, treatment, MSI status, etc.).
- The use of Croissant schemas ensures all processing steps are fully traceable and machine-actionable.

You can continue with more advanced analyses and visualizations, always referencing further fields by their `@id` as needed.